## Homework 3

Homework 3 uses a different dataset, the course lead-scoring data, so the churn notebook is not needed to submit it. The deadline (12 October, 23:00) has passed, but the form is still open for submissions. Questions 1–3 are the same as last year’s. Questions 4–6 have new answer options. I could only see the 2026 questions and options, not the full instructions, because that text requires a login. Before running Q4–Q6, check the model parameters on the logged-in page and adjust the marked lines if they differ. The code below follows the 2025 wording.

### Homework Submission

https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw03

### Setup and data preparation

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv')

categorical = list(df.select_dtypes(exclude='number').columns)
numerical = [c for c in df.select_dtypes(include='number').columns if c != 'converted']

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

### Q1: mode of `industry`

In [4]:
df.industry.value_counts().head()

industry
retail        203
finance       200
other         198
healthcare    187
education     187
Name: count, dtype: int64

### Q2: biggest correlation 

Compare only the four listed pairs

In [5]:
df[numerical].corr()

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.009770,-0.023565,-0.004879
annual_income,0.009770,1.000000,0.027036,0.015610
interaction_count,-0.023565,0.027036,1.000000,0.009888
lead_score,-0.004879,0.015610,0.009888,1.000000


**Split 60/20/20 with seed 42**. Extract the target before deleting it, which is the lesson from the last error.

In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.converted.values
y_val = df_val.converted.values

df_train = df_train.drop(columns='converted').reset_index(drop=True)
df_val = df_val.drop(columns='converted').reset_index(drop=True)

### Q3: Biggest Mutual Information (MI)

Training set only

In [7]:
{c: round(mutual_info_score(df_train[c], y_train), 2) for c in categorical}

{'lead_source': 0.04,
 'industry': 0.01,
 'employment_status': 0.01,
 'location': 0.0}

### Q4: accuracy

In [8]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_tr = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_v = dv.transform(df_val[features].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)  # check against 2026 page
    model.fit(X_tr, y_train)
    return accuracy_score(y_val, model.predict(X_v))

features = categorical + numerical
base_acc = train_and_score(features)
round(base_acc, 2)

0.7

In [11]:
base_acc

0.6996587030716723

### Q5: feature elimination 

Use the unrounded base_acc

In [9]:
for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    acc = train_and_score([x for x in features if x != f])
    print(f, base_acc - acc)

lead_source -0.0034129692832765013
number_of_courses_viewed 0.14334470989761094
interaction_count 0.14334470989761094


The feature with the smallest difference is the least useful. A negative difference is allowed.

### Q6: tuning C

In [10]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(train_and_score(features, C=C), 3))

1e-06 0.556
1e-05 0.556
0.0001 0.556
0.001 0.563


If two values of C tie, last year’s rule was to choose the smallest one.

If an answer does not match an option exactly, choose the closest one. If it falls exactly between two options, choose the higher value.

In [12]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, train_and_score(features, C=C), 3)

1e-06 0.5563139931740614 3
1e-05 0.5563139931740614 3
0.0001 0.5563139931740614 3
0.001 0.5631399317406144 3
